# P03｜像素腦讀論文：Transformer

論文：Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin, "Attention Is All You Need", NeurIPS 2017。https://arxiv.org/abs/1706.03762

這本筆記本是 **像素腦的示意版**，不是論文的程式（論文作者的程式是 Tensor2Tensor：https://github.com/tensorflow/tensor2tensor ）：
1. 自己寫論文的縮放點積注意力，實際看看為什麼要除以 √dk。
2. 用 PyTorch 內建的 nn.Transformer 做一個很小的「翻譯機」（2 層、4 個注意力頭，論文是 6 層、8 個頭），題目是把 8 個數字倒過來寫；再拿掉位置編碼，看看會怎樣。

全部用 CPU 就能跑，大約幾分鐘。

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn

## 1. 注意力：每個字決定要看其他字多少

In [ ]:
def attention(Q, K, V, scale=True):          # 論文的式 (1)：softmax(QKᵀ / √dk) V
    s = Q @ K.T
    if scale:
        s = s / math.sqrt(K.shape[-1])
    w = torch.softmax(s, -1)
    return w @ V, w

words = ["貓", "坐", "在", "墊子", "上"]
torch.manual_seed(0)
x = torch.randn(5, 64)                             # 每個字一個 64 維向量（隨機，示意）
Wq, Wk, Wv = torch.randn(3, 64, 64) / 8            # 三組權重：查詢 Q、鍵 K、值 V
out, w = attention(x @ Wq, x @ Wk, x @ Wv)
print("「貓」看每個字的比例：", [f"{v:.2f}" for v in w[0].tolist()], "加起來", f"{w[0].sum():.2f}")
print("輸出還是 5 個字、每個 64 維：", tuple(out.shape))

## 2. 咦？為什麼要除以 √dk

向量越長，點積的數字越大，softmax 會變成幾乎只看一個字。

In [ ]:
torch.manual_seed(0)
for dk in [4, 64, 512]:
    q, k = torch.randn(1000, dk), torch.randn(1000, 10, dk)
    s = (q[:, None] * k).sum(-1)                   # 1000 組、每組 10 個點積
    raw = torch.softmax(s, -1).max(-1).values.mean()
    fix = torch.softmax(s / math.sqrt(dk), -1).max(-1).values.mean()
    print(f"dk={dk}：點積標準差 {s.std():.1f}，最大權重 不除 {raw:.2f}／除以√dk {fix:.2f}")

## 3. 題目：把數字倒過來寫

In [ ]:
V, L = 12, 8                                       # 0~9 是數字，10 = 開始，11 = 結束
def batch(n):
    src = torch.randint(0, 10, (n, L))
    tgt = src.flip(1)                              # 題目：把數字倒過來寫
    start = torch.full((n, 1), 10)
    return src, torch.cat([start, tgt], 1)

src, tgt = batch(1)
print("題目", src[0].tolist(), "→ 答案", tgt[0, 1:].tolist())

## 4. 很小的 Transformer：編碼器＋解碼器＋位置編碼

In [ ]:
def pos_encoding(n, d):                            # 論文 3.5 節：sin 和 cos 的位置編碼
    pos = torch.arange(n)[:, None]
    i = torch.arange(0, d, 2)
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(pos / 10000 ** (i / d))
    pe[:, 1::2] = torch.cos(pos / 10000 ** (i / d))
    return pe

class TinyTransformer(nn.Module):
    def __init__(self, d=64, heads=4, layers=2, use_pos=True):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.use_pos, self.d = use_pos, d
        self.tf = nn.Transformer(d, heads, layers, layers, 4 * d, dropout=0.0, batch_first=True)
        self.out = nn.Linear(d, V)
    def embed(self, t):
        e = self.emb(t) * math.sqrt(self.d)
        return e + pos_encoding(t.shape[1], self.d) if self.use_pos else e
    def forward(self, src, tgt_in):
        mask = self.tf.generate_square_subsequent_mask(tgt_in.shape[1])   # 不能偷看後面的答案
        h = self.tf(self.embed(src), self.embed(tgt_in), tgt_mask=mask)
        return self.out(h)

net = TinyTransformer()
print(sum(p.numel() for p in net.parameters()), "個參數")

## 5. 訓練：有位置編碼 vs 沒有位置編碼

每個模型練 1500 步。你在 Colab 跑，數字可能會有一點點不同。

In [ ]:
def train(use_pos, steps=1500, seed=0):
    torch.manual_seed(seed)
    net = TinyTransformer(use_pos=use_pos)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for step in range(steps):
        src, tgt = batch(64)
        logits = net(src, tgt[:, :-1])
        loss = nn.functional.cross_entropy(logits.reshape(-1, V), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return net

def translate(net, src):                           # 一個字一個字往下寫
    net.eval()
    out = torch.full((src.shape[0], 1), 10)
    with torch.no_grad():
        for _ in range(L):
            nxt = net(src, out)[:, -1].argmax(-1, keepdim=True)
            out = torch.cat([out, nxt], 1)
    return out[:, 1:]

torch.manual_seed(123)
test_src, test_tgt = batch(500)
for use_pos in [True, False]:
    t0 = time.time()
    m = train(use_pos)
    pred = translate(m, test_src)
    acc = (pred == test_tgt[:, 1:]).float().mean().item()
    full = (pred == test_tgt[:, 1:]).all(1).float().mean().item()
    print(f"{'有' if use_pos else '沒有'}位置編碼：每個數字對 {acc:.1%}，整串全對 {full:.1%}，{time.time() - t0:.0f} 秒")
    if use_pos:
        good = m

## 6. 看它寫每個字時在看原文哪裡

In [ ]:
src = torch.tensor([[3, 1, 4, 1, 5, 9, 2, 6]])
print("題目", src[0].tolist(), "→ 翻譯", translate(good, src)[0].tolist())

attn = {}
layer = good.tf.decoder.layers[-1].multihead_attn     # 最後一層「看原文」的注意力
hook = layer.register_forward_hook(lambda m, i, o: attn.update(w=o[1]))
layer.forward = (lambda f: lambda *a, **k: f(*a, **{**k, "need_weights": True, "average_attn_weights": True}))(layer.forward)
with torch.no_grad():
    good(src, torch.cat([torch.tensor([[10]]), translate(good, src)[:, :-1]], 1))
hook.remove()
w = attn["w"][0]
print("寫第 1 個字時，最看哪個位置：", w[0].argmax().item(), "（原文第 8 個是", src[0, 7].item(), "）")
plt.imshow(w, cmap="magma"); plt.xlabel("source"); plt.ylabel("output"); plt.show()

## 🧪 自己玩玩看
- 把題目改成「排序」（`tgt = src.sort(1).values`），注意力圖會長怎樣？
- 把 `heads` 改成 1、`layers` 改成 1，還學得會嗎？
- 把 `L` 改成 16，沒有位置編碼的版本會更慘嗎？

像素腦 @pixelbraintw